# RENER

TODO:
- ~~provare sentence similarity invece che BM25~~ -> funziona peggio di BM25, quantificare
- testare altri dataset NER (ed eventualmente correggere problemi col nostro dataset)
- ~~provare altri modelli~~ -> la qualità delle metriche è proporzionale al numero di parametri del modello
- ~~testare diversi valori di k (numero di esempi estratti dal document store)~~ -> `k=6` sembra il migliore

In [ ]:
%load_ext autoreload 
%autoreload 2

In [ ]:
import json
from sklearn.model_selection import train_test_split

# Load jsonl file into a list of dictionaries
with open("../data/labelled/all_parallel.jsonl") as test_f:
    data = [json.loads(line) for line in test_f]

# Create test split (notice: fixed seed)
train_data, test_data = train_test_split(data, test_size=0.3, random_state=52)

In [ ]:
from util import evaluate_ner, extract_ner_labels
import rich
from tqdm.notebook import tqdm

from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.dataclasses import Document
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever
from haystack_integrations.components.generators.ollama import OllamaGenerator
from haystack.components.rankers import SentenceTransformersSimilarityRanker

from prompts import guidelines_llama3_legal_lite, default_examples

# Create document store
document_store = InMemoryDocumentStore()
docs = [Document(content=d["input"], meta={"output": d["text"]}) for d in train_data]
document_store.write_documents(docs, policy="overwrite")

models = ["llama3.1:8b", "llama3.1:70b"]
top_k_values = [1, 2, 3, 4, 5, 6, 7, 8]

# Create components
retriever = InMemoryBM25Retriever(document_store=document_store)

llm = OllamaGenerator(
    model="llama3.1:70b",
    url="http://localhost:11434",
    generation_kwargs={"temperature": 0},
)

y_true, y_pred, scores = [], [], []
pbar = tqdm(test_data)
for item in pbar:
    # Define query
    query = (
        f"Input: {item['input']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"
    )

    # Run retriever directly (using BM25)
    # retrieval_result = retriever.run(query=query, top_k=6)
    # retrieved_docs = retrieval_result["documents"]

    # Run retriever directly (using sentence similarity)
    ranker = SentenceTransformersSimilarityRanker()
    ranker.warm_up()
    retrieval_result = ranker.run(query=query, documents=docs, top_k=4)
    retrieved_docs = retrieval_result["documents"]

    # Format retrieved docs using llama's prompt format
    context = "".join(
        [
            f"Input: {doc.content}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\nOutput: {doc.meta['output']}<|eot_id|>\n"
            for i, doc in enumerate(retrieved_docs)
        ]
    ).strip()

    # Build prompt
    prompt = guidelines_llama3_legal_lite + default_examples + context + query

    # Run the LLM and print output
    generation_result = llm.run(
        prompt=prompt,
        generation_kwargs={"temperature": 0},
    )
    pred = generation_result["replies"][0]

    item["pred"] = pred.replace("Output: \n", "")

    true = extract_ner_labels(item["text"])
    true = [(w[0].strip().title(), w[1]) for w in true]

    pred = extract_ner_labels(item["pred"])
    pred = [(w[0].strip().title(), w[1]) for w in pred]

    y_true.append(true)
    y_pred.append(pred)
    scores.append(evaluate_ner(true, pred))

    avg_metrics = {key: sum(d[key] for d in scores) / len(scores) for key in scores[0]}
    pbar.set_postfix(avg_metrics)

    # print("--- LLM Response ---")
    # print(query)
    # print()
    # print(pred)
    # print()

In [ ]:
from itertools import chain
from util import extract_ner_labels
from util import evaluate_ner

# Flatten lists to evaluate metrics on whole dataset
flat_preds = list(map(tuple, chain.from_iterable(y_pred)))
flat_golds = list(map(tuple, chain.from_iterable(y_true)))

print("--- NER evaluation results ---")
results = evaluate_ner(
    flat_golds,
    flat_preds,
)
rich.print(results)

print("--- Token-level evaluation results ---")
tokenlevel_results = evaluate_ner(
    [w[0] for w in flat_golds],
    [w[0] for w in flat_preds],
)
rich.print(tokenlevel_results)

print("--- Macro-average evaluation ---")
scores = [evaluate_ner(y_true[idx], y_pred[idx]) for idx in range(len(y_true))]
avg_metrics = {key: sum(d[key] for d in scores) / len(scores) for key in scores[0]}
rich.print(avg_metrics)